![JohnSnowLabs](https://nlp.johnsnowlabs.com/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp/blob/master/examples/python/annotation/text/english/text-similarity/bm25-retrieval/BM25_Retrieval.ipynb)

# BM25 Retrieval with Spark NLP

**BM25** (Okapi BM25) is a classic *lexical* ranking function used by search engines such as Lucene and Elasticsearch. It scores how relevant each document in a corpus is to a query, based purely on the query terms that appear in the document.

Because a document's score depends on **corpus-level statistics** (how many documents contain a term, and the average document length), BM25 is implemented in Spark NLP as a two-phase **Estimator/Model** pair:

* `BM25Approach`: scans the corpus once during `fit()` and learns the document count `N`, the document frequency `df(t)` of every term, the average document length `avgdl`, and the inverse document frequency `idf(t)`.
* `BM25Model`: reuses those statistics at query time to score every document against a user-provided query.

Spark NLP uses the non-negative (Lucene/Elasticsearch) IDF variant:

$$ idf(t) = \ln\left(1 + \frac{N - df(t) + 0.5}{df(t) + 0.5}\right) $$

and scores each document `D` against query `Q` as:

$$ score(D, Q) = \sum_{t \in Q} idf(t) \cdot \frac{tf(t, D) \cdot (k_1 + 1)}{tf(t, D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{avgdl}\right)} $$

| Parameter | Meaning | Default | Typical range |
|-----------|---------|---------|---------------|
| `k1` | Term-frequency saturation | `1.2` | `[1.0, 2.0]` |
| `b` | Document-length normalization | `0.75` | `[0.0, 1.0]` |
| `minDocFreq` | Drop terms seen in fewer than N docs | `1` | |
| `caseSensitive` | Treat tokens case-sensitively | `False` | |

The input is a column of `TOKEN` annotations, so BM25 is placed after a `Tokenizer` (optionally a `Normalizer` and/or `StopWordsCleaner`).

In [ ]:
# Only run this cell when you are using Spark NLP on Google Colab
!wget http://setup.johnsnowlabs.com/colab.sh -O - | bash

In [ ]:
import sparknlp
from sparknlp.base import *
from sparknlp.annotator import *
from pyspark.ml import Pipeline
from pyspark.sql.functions import col, explode

spark = sparknlp.start()

print("Spark NLP version:", sparknlp.version())
print("Apache Spark version:", spark.version)

## The corpus

A small toy corpus of 10 documents spanning a few topics (nutrition, machine learning, travel).

In [8]:
corpus = spark.createDataFrame([
    (1,  "Apples are a great source of dietary fiber and vitamin C."),
    (2,  "Bananas are rich in potassium and natural sugars."),
    (3,  "Machine learning is a subset of artificial intelligence."),
    (4,  "Deep learning uses neural networks with many layers."),
    (5,  "Florence in Italy is one of the most beautiful cities in Europe."),
    (6,  "The French Riviera is a warm coastal region in southern France."),
    (7,  "Vitamin C deficiency can lead to scurvy and immune system problems."),
    (8,  "Neural networks are inspired by the human brain's structure."),
    (9,  "Potassium is an essential mineral for heart and muscle function."),
    (10, "Italy is home to some of the world's greatest Renaissance art."),
], ["id", "text"])

corpus.show(truncate=False)

+---+-------------------------------------------------------------------+
|id |text                                                               |
+---+-------------------------------------------------------------------+
|1  |Apples are a great source of dietary fiber and vitamin C.          |
|2  |Bananas are rich in potassium and natural sugars.                  |
|3  |Machine learning is a subset of artificial intelligence.           |
|4  |Deep learning uses neural networks with many layers.               |
|5  |Florence in Italy is one of the most beautiful cities in Europe.   |
|6  |The French Riviera is a warm coastal region in southern France.    |
|7  |Vitamin C deficiency can lead to scurvy and immune system problems.|
|8  |Neural networks are inspired by the human brain's structure.       |
|9  |Potassium is an essential mineral for heart and muscle function.   |
|10 |Italy is home to some of the world's greatest Renaissance art.     |
+---+---------------------------------

## 1. Basic BM25 document ranking with a global query

Build the training pipeline (`DocumentAssembler` &rarr; `Tokenizer` &rarr; `StopWordsCleaner` &rarr; `BM25Approach`), `fit` it on the corpus to learn the IDF/avgdl statistics, set a query, then `transform`.

In [9]:
document_assembler = (
    DocumentAssembler()
    .setInputCol("text")
    .setOutputCol("document")
)

tokenizer = (
    Tokenizer()
    .setInputCols(["document"])
    .setOutputCol("token")
)

stop_words_cleaner = (
    StopWordsCleaner()
    .setInputCols(["token"])
    .setOutputCol("clean_token")
    .setCaseSensitive(False)
)

bm25 = (
    BM25Approach()
    .setInputCols(["clean_token"])
    .setOutputCol("bm25_rankings")
    .setK1(1.2)        # TF saturation   [1.0 - 2.0]
    .setB(0.75)        # length norm     [0.0 - 1.0]
    .setMinDocFreq(1)  # drop terms seen in fewer than N docs
    .setCaseSensitive(False)
)

fit_pipeline = Pipeline(stages=[
    document_assembler,
    tokenizer,
    stop_words_cleaner,
    bm25,
])

# FIT: scan corpus once, learn IDF + avgdl
pipeline_model = fit_pipeline.fit(corpus)

bm25_model = pipeline_model.stages[-1]
print("N (documents):", bm25_model.getNumDocuments())
print("avgdl        :", round(bm25_model.getAvgDocLength(), 3))

N (documents): 10
avgdl        : 7.3


In [10]:
def ranked(model, df):
    """Explode the bm25_rankings annotation and sort documents by score."""
    return (
        model.transform(df)
        .select(
            col("id"),
            col("text"),
            explode(col("bm25_rankings")).alias("ranking"),
        )
        .select(
            col("id"),
            col("text"),
            col("ranking.metadata")["bm25_score"].cast("double").alias("bm25_score"),
            col("ranking.metadata")["num_query_terms_matched"].cast("int").alias("terms_matched"),
        )
        .orderBy(col("bm25_score").desc())
    )


bm25_model.setQuery("vitamin C health benefits fruits")
ranked(pipeline_model, corpus).show(truncate=False)

+---+-------------------------------------------------------------------+------------------+-------------+
|id |text                                                               |bm25_score        |terms_matched|
+---+-------------------------------------------------------------------+------------------+-------------+
|1  |Apples are a great source of dietary fiber and vitamin C.          |2.8513563723478614|2            |
|7  |Vitamin C deficiency can lead to scurvy and immune system problems.|2.705465483484128 |2            |
|2  |Bananas are rich in potassium and natural sugars.                  |0.0               |0            |
|6  |The French Riviera is a warm coastal region in southern France.    |0.0               |0            |
|3  |Machine learning is a subset of artificial intelligence.           |0.0               |0            |
|8  |Neural networks are inspired by the human brain's structure.       |0.0               |0            |
|4  |Deep learning uses neural networ

The two documents mentioning *vitamin C* (ids 1 and 7) rise to the top, documents with no query term score exactly `0.0`, and `terms_matched` reports how many distinct query terms occurred in each document.

## 2. Saving and loading a fitted BM25 model

A fitted `BM25Model` carries the learned IDF map, `avgdl` and document count. It can be persisted once and reloaded later for many queries, no need to re-scan the corpus.

In [11]:
bm25_model.write().overwrite().save("/tmp/bm25_corpus_model")

loaded_bm25 = BM25Model.load("/tmp/bm25_corpus_model")
loaded_bm25.setQuery("neural networks deep learning")

print("Reloaded N (documents):", loaded_bm25.getNumDocuments())

infer_pipeline = Pipeline(stages=[
    document_assembler,
    tokenizer,
    stop_words_cleaner,
    loaded_bm25,
])

infer_model = infer_pipeline.fit(corpus)
ranked(infer_model, corpus).show(truncate=False)

Reloaded N (documents): 10
+---+-------------------------------------------------------------------+------------------+-------------+
|id |text                                                               |bm25_score        |terms_matched|
+---+-------------------------------------------------------------------+------------------+-------------+
|4  |Deep learning uses neural networks with many layers.               |6.194256154982231 |4            |
|8  |Neural networks are inspired by the human brain's structure.       |3.0138782681751617|2            |
|3  |Machine learning is a subset of artificial intelligence.           |1.5980234336630559|1            |
|6  |The French Riviera is a warm coastal region in southern France.    |0.0               |0            |
|1  |Apples are a great source of dietary fiber and vitamin C.          |0.0               |0            |
|7  |Vitamin C deficiency can lead to scurvy and immune system problems.|0.0               |0            |
|2  |Banan

## 3. Reusing the same fitted model with different queries

BM25 is a "fit once, query many times" workflow. The IDF and `avgdl` statistics are computed only during `fit`; calling `setQuery` again simply re-scores the documents with the existing statistics.

In [12]:
loaded_bm25.setQuery("vitamin C health benefits")
print("Query: vitamin C health benefits")
ranked(infer_model, corpus).show(truncate=False)

loaded_bm25.setQuery("neural networks artificial intelligence")
print("Query: neural networks artificial intelligence")
ranked(infer_model, corpus).show(truncate=False)

loaded_bm25.setQuery("Italy France Europe travel")
print("Query: Italy France Europe travel")
ranked(infer_model, corpus).show(truncate=False)

Query: vitamin C health benefits
+---+-------------------------------------------------------------------+------------------+-------------+
|id |text                                                               |bm25_score        |terms_matched|
+---+-------------------------------------------------------------------+------------------+-------------+
|1  |Apples are a great source of dietary fiber and vitamin C.          |2.8513563723478614|2            |
|7  |Vitamin C deficiency can lead to scurvy and immune system problems.|2.705465483484128 |2            |
|2  |Bananas are rich in potassium and natural sugars.                  |0.0               |0            |
|6  |The French Riviera is a warm coastal region in southern France.    |0.0               |0            |
|3  |Machine learning is a subset of artificial intelligence.           |0.0               |0            |
|8  |Neural networks are inspired by the human brain's structure.       |0.0               |0            |
|4  

The same fitted `infer_model` produces a different ranking for each query, while reusing the identical IDF/`avgdl` statistics learned during the single `fit`.

## 4. Ranking with preprocessing

BM25 plugs into a normal Spark NLP pipeline. Here we add a `Normalizer` (to strip punctuation and lowercase) before stop-word removal, so the tokens fed to BM25 are cleaner.

In [13]:
document_assembler = (
    DocumentAssembler()
    .setInputCol("text")
    .setOutputCol("document")
)

tokenizer = (
    Tokenizer()
    .setInputCols(["document"])
    .setOutputCol("token")
)

normalizer = (
    Normalizer()
    .setInputCols(["token"])
    .setOutputCol("normalized")
    .setLowercase(True)
)

stop_words_cleaner = (
    StopWordsCleaner()
    .setInputCols(["normalized"])
    .setOutputCol("clean_token")
    .setCaseSensitive(False)
)

bm25 = (
    BM25Approach()
    .setInputCols(["clean_token"])
    .setOutputCol("bm25_rankings")
)

preprocess_pipeline = Pipeline(stages=[
    document_assembler,
    tokenizer,
    normalizer,
    stop_words_cleaner,
    bm25,
])

preprocess_model = preprocess_pipeline.fit(corpus)
preprocess_model.stages[-1].setQuery("vitamin C health benefits fruits")
ranked(preprocess_model, corpus).show(truncate=False)

+---+-------------------------------------------------------------------+------------------+-------------+
|id |text                                                               |bm25_score        |terms_matched|
+---+-------------------------------------------------------------------+------------------+-------------+
|1  |Apples are a great source of dietary fiber and vitamin C.          |2.834373904376761 |2            |
|7  |Vitamin C deficiency can lead to scurvy and immune system problems.|2.6686210444716867|2            |
|2  |Bananas are rich in potassium and natural sugars.                  |0.0               |0            |
|6  |The French Riviera is a warm coastal region in southern France.    |0.0               |0            |
|3  |Machine learning is a subset of artificial intelligence.           |0.0               |0            |
|8  |Neural networks are inspired by the human brain's structure.       |0.0               |0            |
|4  |Deep learning uses neural networ

## 5. Keep the query analyzer in sync (`setQueryTokens`)

`setQuery("...")` is a convenience: the model tokenizes the string itself by splitting on non-word characters and lowercasing. That stays correct only while the corpus pipeline doesn't *transform* tokens beyond that.

Once a stage rewrites tokens — a **stemmer**, **lemmatizer**, or aggressive **`Normalizer`** — the learned IDF vocabulary holds the *transformed* forms, but a raw-string query is **not** transformed, so its terms silently fail to match and contribute nothing.

The robust pattern is to analyze the query with the **same** stages used for the documents (for example with a `LightPipeline`) and pass the resulting tokens to `setQueryTokens(...)`; when set, `queryTokens` overrides `query`. Below, a Porter-stemmed corpus only matches `"deficiency"` when the query is stemmed the same way.

In [14]:
from sparknlp.base import LightPipeline

# A token-transforming stage (here a Porter stemmer) is where a raw-string query breaks:
# the corpus vocabulary holds stems, but setQuery only splits on non-word chars + lowercases.
stemmer = (
    Stemmer()
    .setInputCols(["token"])
    .setOutputCol("stem")
)

stem_bm25 = (
    BM25Approach()
    .setInputCols(["stem"])
    .setOutputCol("bm25_rankings")
)

stem_model = Pipeline(stages=[
    document_assembler,
    tokenizer,
    stemmer,
    stem_bm25,
]).fit(corpus)
stem_bm25_model = stem_model.stages[-1]

# (a) Raw-string query: "deficiency" is never stemmed, so it misses the stemmed vocabulary.
stem_bm25_model.setQuery("deficiency")
print('Raw setQuery("deficiency"):')
ranked(stem_model, corpus).select("id", "bm25_score", "terms_matched").show(3, truncate=False)

# (b) Analyze the query with the SAME stages, then pass the tokens via setQueryTokens.
analyzer = Pipeline(stages=[document_assembler, tokenizer, stemmer]).fit(corpus)
query_tokens = LightPipeline(analyzer).annotate("deficiency")["stem"]
print("Analyzed query tokens:", query_tokens)

stem_bm25_model.setQueryTokens(query_tokens)
print("setQueryTokens(query_tokens):")
ranked(stem_model, corpus).select("id", "bm25_score", "terms_matched").show(3, truncate=False)

Raw setQuery("deficiency"):
+---+----------+-------------+
|id |bm25_score|terms_matched|
+---+----------+-------------+
|6  |0.0       |0            |
|1  |0.0       |0            |
|7  |0.0       |0            |
+---+----------+-------------+
only showing top 3 rows

Analyzed query tokens: ['defici']
setQueryTokens(query_tokens):
+---+------------------+-------------+
|id |bm25_score        |terms_matched|
+---+------------------+-------------+
|7  |1.9134351361342068|1            |
|1  |0.0               |0            |
|6  |0.0               |0            |
+---+------------------+-------------+
only showing top 3 rows



## Notes & limitations

* **Purely lexical.** BM25 has no notion of synonyms or semantics, *car* and *automobile* are unrelated terms. For semantic ranking, see `DocumentSimilarityRankerApproach` (embedding + LSH based).
* **Vocabulary drift.** If the corpus grows significantly after the model is fit, the IDF map becomes stale and the model should be refit.
* **Annotation output.** Each document yields a single `bm25_rankings` annotation whose `result` is the score and whose `metadata` contains `bm25_score`, `num_query_terms_matched`, `query` and `doc_len`.
* **Analyzer symmetry.** `setQuery` only splits on non-word characters and lowercases; if your pipeline transforms tokens (stemming, lemmatization, aggressive normalization) the learned vocabulary holds the transformed forms, so analyze the query through the same pipeline and pass the result to `setQueryTokens` — otherwise query terms can silently miss the vocabulary and score 0.